# Step 8b: Mini REPL — 중간 통합

## 학습 목표

- Step 1~8까지 구현한 서브시스템을 **하나의 동작하는 REPL**로 통합합니다
- **커맨드 vs 일반 메시지**의 분기 처리를 구현합니다
- 실제로 대화하며 도구를 호출하는 **대화형 에이전트**를 실행합니다

> **왜 중간 통합인가?**
> 지금까지 8개의 서브시스템을 개별적으로 만들었습니다.
> 하지만 개별 부품이 잘 동작하는 것과, 조립된 전체가 잘 동작하는 것은 다릅니다.
> Step 13(최종 통합) 전에, 핵심 부품만으로 동작하는 미니 버전을 먼저 만들어 봅니다.
> 이것이 Claude Code의 `src/entrypoints/cli.tsx`의 축소판입니다.

---

## Claude Code 소스 분석 — CLI 진입점

### 8b-1. CLI 부트 시퀀스 (src/entrypoints/cli.tsx)

Claude Code의 진입점은 `src/entrypoints/cli.tsx`입니다. 이 파일은 **빠른 경로(fast-path)** 패턴을 사용하여, 특별한 플래그(`--version`, `--daemon`, `--bridge` 등)를 먼저 처리하고, 나머지는 `src/main.tsx`로 위임합니다:

```typescript
// src/entrypoints/cli.tsx — 부트 시퀀스 (간소화)

async function main(): Promise<void> {
  const args = process.argv.slice(2)

  // Fast-path 1: --version (모듈 로딩 없이 즉시 출력)
  if (args[0] === '--version') { console.log(MACRO.VERSION); return }

  // Fast-path 2: --daemon-worker (경량 워커 실행)
  if (args[0] === '--daemon-worker') { await runDaemonWorker(args[1]); return }

  // Fast-path 3: bridge (원격 제어 모드)
  if (args[0] === 'remote-control') { await bridgeMain(args.slice(1)); return }

  // ... 기타 fast-path들 ...

  // 기본 경로: 전체 CLI 로드
  const { main: cliMain } = await import('../main.js')
  await cliMain()
}
```

**설계 포인트:** `--version`처럼 간단한 요청은 무거운 모듈을 로드하지 않고 즉시 처리합니다. 이것이 "fast-path" 패턴입니다. 우리의 MiniREPL에서도 `/quit`, `/help` 같은 로컬 커맨드가 이 역할을 합니다.

### 8b-2. 사용자 입력 처리 흐름 (전체 그림)

`src/main.tsx`에서 REPL이 시작된 후, 사용자 입력은 다음 흐름을 따릅니다:

```
사용자 입력
  ├── /커맨드? → 커맨드 파싱
  │   ├── LocalCommand → 즉시 실행 → 결과 표시
  │   └── PromptCommand → 프롬프트 생성 → QueryEngine
  └── 일반 텍스트 → QueryEngine
                        ↓
                    while(true) {
                      API 호출 → 도구 호출? → 권한 체크 → 도구 실행 → 반복
                    }
                        ↓
                    최종 응답 표시
```

이 흐름에서 관여하는 서브시스템들:

| 단계 | 서브시스템 | Step |
|------|-----------|------|
| 입력 분기 | REPL (커맨드 파서) | 5 |
| API 호출 | LLMClient | 1 |
| 에이전트 루프 | agent_loop (while true) | 1 |
| 도구 관리 | ToolRegistry | 2 |
| 권한 체크 | PermissionSystem | 7 |
| 사전/사후 처리 | HookRegistry | 6 |
| 컨텍스트 | ContextManager | 5 |

### 8b-3. 커맨드 시스템의 세 가지 타입 (src/commands.ts)

Claude Code의 커맨드는 세 가지 타입으로 나뉩니다:

```typescript
// src/commands.ts — 커맨드 타입

type PromptCommand = {
  type: 'prompt'
  // 사용자 입력을 프롬프트로 변환 → QueryEngine에 전달
  getPrompt(args: string): string
}

type LocalCommand = {
  type: 'local'
  // 즉시 실행하고 텍스트 결과 반환 (API 호출 없음)
  execute(args: string): string
}

type LocalJSXCommand = {
  type: 'local-jsx'
  // 즉시 실행하고 JSX 컴포넌트 반환 (터미널 UI)
  execute(args: string): JSX.Element
}
```

**핵심:** `LocalCommand`는 API 호출 없이 즉시 실행됩니다 (`/help`, `/cost`, `/clear` 등). `PromptCommand`는 입력을 변환한 후 QueryEngine에 전달합니다 (`/review`, `/explain` 등). 이 분기가 REPL의 핵심 설계입니다.

---

## Python 구현 — MiniQueryEngine

Step 1~8까지의 서브시스템을 하나로 통합하는 쿼리 엔진입니다.

```
MiniQueryEngine 통합 요소:
  - LLMClient (Step 1)        — API 호출
  - ToolRegistry (Step 2)     — 도구 관리 및 스키마 생성
  - PermissionSystem (Step 7) — 권한 체크
  - HookRegistry (Step 6)     — 사전/사후 훅
```

Claude Code의 `queryLoop()` (src/query.ts:307)과 도구 실행 파이프라인(src/services/tools/toolExecution.ts)을 하나의 클래스로 축소한 것입니다.

In [ ]:
import sys, os
sys.path.insert(0, ".")

from mini_claude.llm_client import (
    create_client, LLMClient, LLMResponse, ToolCall,
    StreamEvent, tool_result_message, AnthropicClient,
)
from mini_claude.tool_base import Tool, ToolResult
from mini_claude.tool_registry import ToolRegistry
from mini_claude.permissions import (
    PermissionSystem, PermissionMode, PermissionResult,
    PermissionCheckResult, create_default_rules,
)

print("mini_claude 모듈 임포트 완료")
print(f"  - LLMClient, LLMResponse, ToolCall")
print(f"  - Tool, ToolResult, ToolRegistry")
print(f"  - PermissionSystem, PermissionMode, PermissionResult")

In [ ]:
class MiniQueryEngine:
    """
    Step 1~8의 서브시스템을 통합한 미니 쿼리 엔진.

    Claude Code의 queryLoop() (src/query.ts:307)에 대응합니다.
    실제 Claude Code에서는 AsyncGenerator로 이벤트를 yield하지만,
    여기서는 단순화를 위해 최종 텍스트를 반환합니다.

    통합 요소:
    - LLMClient (Step 1)        — API 호출
    - ToolRegistry (Step 2)     — 도구 관리
    - PermissionSystem (Step 7) — 권한 체크
    """

    def __init__(
        self,
        client: LLMClient,
        tool_registry: ToolRegistry,
        permission_system: PermissionSystem | None = None,
        system_prompt: str = "",
        max_turns: int = 20,
    ):
        self.client = client
        self.tool_registry = tool_registry
        self.permissions = permission_system or PermissionSystem(mode=PermissionMode.ASK)
        self.system_prompt = system_prompt
        self.max_turns = max_turns

        # 세션 상태
        self.messages: list[dict] = []
        self.total_input_tokens = 0
        self.total_output_tokens = 0

    def _get_tools_schema(self) -> list[dict] | None:
        """클라이언트 타입에 맞는 도구 스키마 생성"""
        if len(self.tool_registry) == 0:
            return None
        if isinstance(self.client, AnthropicClient):
            return self.tool_registry.to_api_schemas("anthropic")
        else:
            return self.tool_registry.to_api_schemas("openai")

    async def submit(self, user_message: str, verbose: bool = True) -> str:
        """
        사용자 메시지를 처리하고 최종 응답 반환.

        이것이 Claude Code의 queryLoop()에 대응하는 핵심 메서드입니다:
        1. 사용자 메시지를 히스토리에 추가
        2. while 루프: API 호출 → 도구 호출 여부 판단 → 도구 실행 → 반복
        3. 도구 호출 없으면 → 최종 텍스트 반환
        """
        self.messages.append({"role": "user", "content": user_message})

        tools_schema = self._get_tools_schema()

        for turn in range(1, self.max_turns + 1):
            if verbose:
                print(f"\n--- Turn {turn} ---")

            # 1. API 호출 (query.ts:307의 stream() 호출에 대응)
            response = await self.client.query(
                messages=self.messages,
                system=self.system_prompt,
                tools=tools_schema,
                max_tokens=4096,
            )

            self.total_input_tokens += response.input_tokens
            self.total_output_tokens += response.output_tokens

            # 2. 도구 호출이 없으면 → 최종 응답 (query.ts:1062의 !needsFollowUp)
            if not response.has_tool_calls:
                if response.text:
                    self.messages.append({"role": "assistant", "content": response.text})
                if verbose:
                    print(f"  최종 응답 (토큰: in={self.total_input_tokens:,}, out={self.total_output_tokens:,})")
                return response.text

            # 3. assistant 메시지를 히스토리에 추가
            assistant_msg = {
                "role": "assistant",
                "text": response.text,
                "tool_calls": [
                    {"id": tc.id, "name": tc.name, "input": tc.input}
                    for tc in response.tool_calls
                ],
            }
            self.messages.append(assistant_msg)

            # 4. 도구 실행 (권한 체크 포함 — toolExecution.ts에 대응)
            for tc in response.tool_calls:
                # Stage 1: 권한 체크 (permissions.check_permission)
                tool = self.tool_registry.find_by_name(tc.name)
                is_read_only = tool.is_read_only() if tool else False

                perm_result = await self.permissions.check_permission(
                    tc.name, tc.input, is_read_only=is_read_only
                )

                if perm_result.result == PermissionResult.DENIED:
                    result = f"권한 거부: {perm_result.reason}"
                    if verbose:
                        print(f"  [차단] {tc.name} -- {perm_result.reason}")
                else:
                    # Stage 2: 도구 실행
                    if tool:
                        try:
                            result = await self.tool_registry.execute(tc.name, tc.input)
                            if verbose:
                                result_preview = str(result)[:150]
                                print(f"  [도구] {tc.name}({tc.input}) -> {result_preview}")
                        except Exception as e:
                            result = f"Error: {e}"
                    else:
                        result = f"알 수 없는 도구: {tc.name}"

                self.messages.append(tool_result_message(tc.id, str(result)))

        return "[max turns reached]"

    def clear(self):
        """대화 히스토리 초기화"""
        self.messages.clear()
        self.total_input_tokens = 0
        self.total_output_tokens = 0


print("MiniQueryEngine 정의 완료")
print()
print("핵심 메서드:")
print("  submit(message)  — 사용자 메시지 처리 (에이전트 루프)")
print("  clear()          — 대화 히스토리 초기화")

---

## MiniREPL 구현

MiniREPL은 Claude Code의 REPL(Read-Eval-Print Loop)을 축소 구현합니다.

핵심 책임은 **입력 분기**입니다:
- `/`로 시작하는 입력 → 커맨드 파싱 → `LocalCommand`면 즉시 실행, `PromptCommand`면 QueryEngine에 전달
- 그 외 → `MiniQueryEngine.submit()`에 직접 전달

이 단순한 분기가 전체 시스템의 진입점입니다.

### 커맨드 타입 정의

Claude Code의 `src/commands.ts`에서 세 가지 커맨드 타입을 정의하듯이,
여기서는 `LocalCommand`와 `PromptCommand` 두 가지만 사용합니다.

In [ ]:
from abc import ABC, abstractmethod


# --- 커맨드 시스템 (Step 5에서 구현한 것의 축소판) ---

class Command(ABC):
    """커맨드 기반 클래스 — Claude Code의 Command 타입에 대응"""
    name: str = ""
    description: str = ""
    aliases: list[str] = []


class PromptCommand(Command):
    """프롬프트를 생성하여 QueryEngine에 전달하는 커맨드"""
    @abstractmethod
    def get_prompt(self, args: str) -> str: ...


class LocalCommand(Command):
    """즉시 실행하고 결과를 반환하는 커맨드 (API 호출 없음)"""
    @abstractmethod
    def execute(self, args: str) -> str: ...


# --- 내장 커맨드들 ---

class CostCommand(LocalCommand):
    """현재 세션의 토큰 사용량과 비용 표시 — Claude Code의 /cost에 대응"""
    name = "cost"
    description = "현재 세션의 토큰 사용량과 비용 표시"

    def __init__(self, get_engine):
        self._get_engine = get_engine

    def execute(self, args: str) -> str:
        engine = self._get_engine()
        if not engine:
            return "활성 세션 없음"
        cost = (engine.total_input_tokens * 3.0 + engine.total_output_tokens * 15.0) / 1_000_000
        return (
            f"세션 비용 요약\n"
            f"  입력 토큰: {engine.total_input_tokens:>10,}\n"
            f"  출력 토큰: {engine.total_output_tokens:>10,}\n"
            f"  추정 비용: ${cost:.4f}"
        )


class ClearCommand(LocalCommand):
    """대화 히스토리 초기화 — Claude Code의 /clear에 대응"""
    name = "clear"
    description = "���화 히스토리 초기화"
    aliases = ["cl"]

    def __init__(self, get_engine):
        self._get_engine = get_engine

    def execute(self, args: str) -> str:
        engine = self._get_engine()
        if engine:
            count = len(engine.messages)
            engine.clear()
            return f"대화 히스토리 {count}개 메시지 삭제됨"
        return "활성 세션 없음"


class ToolsCommand(LocalCommand):
    """등록된 도구 목록 표시 — Claude Code의 /tools에 대응"""
    name = "tools"
    description = "등��된 도구 목��� 표시"

    def __init__(self, get_registry):
        self._get_registry = get_registry

    def execute(self, args: str) -> str:
        registry = self._get_registry()
        names = registry.get_names()
        lines = [f"등록된 도구 ({len(names)}개):\n"]
        for name in names:
            tool = registry.find_by_name(name)
            desc = tool.description[:50] if tool else ""
            lines.append(f"  {name:15s} -- {desc}")
        return "\n".join(lines)


class HelpCommand(LocalCommand):
    """사용 가능한 커맨드 목록 표시 — Claude Code의 /help에 대응"""
    name = "help"
    description = "사용 가능한 ���맨드 ��록 표시"
    aliases = ["h", "?"]

    def __init__(self, get_commands):
        self._get_commands = get_commands

    def execute(self, args: str) -> str:
        commands = self._get_commands()
        lines = ["사용 가능한 커맨드:\n"]
        for cmd in commands:
            alias_str = f" ({', '.join('/' + a for a in cmd.aliases)})" if cmd.aliases else ""
            cmd_type = "Prompt" if isinstance(cmd, PromptCommand) else "Local"
            lines.append(f"  /{cmd.name:<12} [{cmd_type:>6}] {cmd.description}{alias_str}")
        lines.append("\n/가 없는 메시지는 AI에게 직접 전달됩니다.")
        return "\n".join(lines)


print("커맨드 타입 정의 완료:")
print("  Command (기반), LocalCommand, PromptCommand")
print("  CostCommand, ClearCommand, ToolsCommand, HelpCommand")

In [ ]:
class MiniREPL:
    """
    Claude Code의 CLI REPL을 축소 구현.

    입력 흐름:
      /커맨드 -> 커맨드 파싱 -> LocalCommand면 즉시 실행
                             -> PromptCommand면 프롬프트를 QueryEngine에 전달
      일반 메시지 -> QueryEngine에 직접 전달

    이것이 Claude Code의 src/ink/components/App.tsx의 processInput()에 대응합니다.
    """

    def __init__(self, engine: MiniQueryEngine):
        self.engine = engine
        self._commands: dict[str, Command] = {}
        self._aliases: dict[str, str] = {}

        # 기본 커맨드 등록
        self.register(CostCommand(lambda: self.engine))
        self.register(ClearCommand(lambda: self.engine))
        self.register(ToolsCommand(lambda: self.engine.tool_registry))
        self.register(HelpCommand(lambda: list(self._commands.values())))

    def register(self, cmd: Command):
        """커맨드 등록 — Claude Code의 registerCommand()에 대응"""
        self._commands[cmd.name] = cmd
        for alias in getattr(cmd, "aliases", []):
            self._aliases[alias] = cmd.name

    def parse_input(self, text: str) -> tuple[Command | None, str]:
        """
        사용자 입력 파싱 -> (커맨드, 인자) 또는 (None, 원본 텍스트)

        '/'로 시작하면 커맨드로 파싱, 아니면 일반 텍스트.
        이 단순한 분기가 REPL의 핵심 설계입니다.
        """
        if not text.startswith("/"):
            return None, text

        parts = text[1:].split(" ", 1)
        cmd_name = parts[0].lower()
        args = parts[1] if len(parts) > 1 else ""

        # 별칭 해석
        if cmd_name in self._aliases:
            cmd_name = self._aliases[cmd_name]

        return self._commands.get(cmd_name), args

    async def process(self, user_input: str, verbose: bool = True) -> str:
        """
        사용자 입력을 처리하고 결과 반환.

        이것이 REPL의 핵심 메서드입니다:
        1. 입력 파싱 (커맨드 vs 일반 텍스트)
        2. 커맨드면 -> 타입에 따라 즉시 실행 또는 QueryEngine 전달
        3. 일반 텍스트면 -> QueryEngine.submit() 호출
        """
        cmd, args = self.parse_input(user_input)

        if cmd is None:
            # 일반 메시지 -> QueryEngine
            return await self.engine.submit(user_input, verbose=verbose)

        if isinstance(cmd, LocalCommand):
            # LocalCommand -> 즉시 실행 (API 호출 없음, fast-path!)
            return cmd.execute(args)

        if isinstance(cmd, PromptCommand):
            # PromptCommand -> 프롬프트 변환 -> QueryEngine
            prompt = cmd.get_prompt(args)
            return await self.engine.submit(prompt, verbose=verbose)

        return f"알 수 없는 커맨드: {user_input}"

    async def run(self):
        """
        대화형 REPL 루프 (Jupyter에서는 process()를 직접 호출).

        이것이 "외부 루프"입니다:
          외부 루프 (REPL: 사용자 입력 반복)
            -> 내부 루프 (agent_loop: API 호출 반복)
        """
        print("Mini Claude REPL (종료: /quit)")
        print("도움말: /help\n")

        while True:
            try:
                user_input = input("You> ").strip()
            except (EOFError, KeyboardInterrupt):
                print("\n종료합니다.")
                break

            if not user_input:
                continue
            if user_input.lower() in ("/quit", "/exit", "/q"):
                print("종료합니다.")
                break

            result = await self.process(user_input)
            print(f"\n{result}\n")


print("MiniREPL 정의 완료")
print()
print("구조:")
print("  MiniREPL")
print("    +-- parse_input()  : '/' 분기 처리")
print("    +-- process()      : 단일 입력 처리")
print("    +-- run()          : 대화형 루프")
print("    +-- MiniQueryEngine")
print("          +-- submit() : 에이전트 루프 (while true)")
print("          +-- ToolRegistry, PermissionSystem, LLMClient")

---

## 실습 — Mini REPL 데모

MiniREPL을 실제로 구성하고 실행합니다. 전체 흐름을 확인합니다:

1. **도구 등록** — FileReadTool, BashTool을 ToolRegistry에 등록
2. **권한 설정** — 기본 규칙으로 PermissionSystem 구성
3. **엔진 생성** — MiniQueryEngine에 모든 서브시스템 주입
4. **REPL 생성** — MiniREPL에 엔진 연결, 커맨드 등록
5. **테스트** — LocalCommand, 일반 메시지, 비용 확인

API 키가 없으면 MockClient로 구조만 확인합니다.

In [ ]:
# --- 1. 도구 등록 ---
from mini_claude.tools.file_read_tool import FileReadTool
from mini_claude.tools.bash_tool import BashTool

registry = ToolRegistry()
registry.register(FileReadTool)
registry.register(BashTool)

print("=== 도구 등록 완료 ===")
for name in registry.get_names():
    tool = registry.find_by_name(name)
    ro = "읽기전용" if tool.is_read_only() else "쓰기가능"
    print(f"  {name:10s} [{ro}] {tool.description[:60]}")

# --- 2. 권한 설정 ---
permissions = PermissionSystem(
    mode=PermissionMode.ASK,
    rules=create_default_rules(),
)
print(f"\n=== 권한 시스템 ===")
print(f"  모드: {permissions.mode.value}")
print(f"  규칙: {len(permissions.rules)}개")

In [ ]:
# --- 3. 엔진 + REPL 생성 및 테스트 ---

has_api = bool(os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("OPENAI_API_KEY"))

if has_api:
    client = create_client()
    engine = MiniQueryEngine(
        client=client,
        tool_registry=registry,
        permission_system=permissions,
        system_prompt="당신은 도구를 사용하여 사용자를 돕는 AI 어시스턴트입니다. 한국어로 답변하세요.",
    )
    repl = MiniREPL(engine)

    # --- LocalCommand 테스트 ---
    print("=== /help ===")
    print(await repl.process("/help", verbose=False))

    print(f"\n{'=' * 60}")
    print("\n=== /tools ===")
    print(await repl.process("/tools", verbose=False))

    print(f"\n{'=' * 60}")
    print("\n=== 일반 메시지 (도구 호출 포함) ===")
    result = await repl.process("현재 디렉토리의 파일 목록을 보여줘", verbose=True)
    print(f"\n응답: {result[:300]}")

    print(f"\n{'=' * 60}")
    print("\n=== /cost ===")
    print(await repl.process("/cost", verbose=False))

else:
    print("=== API 키 없이 구조 확인 ===\n")

    # Mock 클라이언트로 구조만 확인
    class MockClient(LLMClient):
        """API 호출 없이 구조를 확인하기 위한 Mock 클라이언트"""
        async def query(self, messages, system="", tools=None, max_tokens=4096):
            return LLMResponse(text="안녕하세요! (mock 응답)", input_tokens=50, output_tokens=20)

        async def stream(self, messages, system="", tools=None, max_tokens=4096):
            yield StreamEvent(type="text_delta", text="mock")

    engine = MiniQueryEngine(
        client=MockClient(),
        tool_registry=registry,
        permission_system=permissions,
        system_prompt="Test prompt",
    )
    repl = MiniREPL(engine)

    # --- LocalCommand 테스트 (API 불필요) ---
    print("=== /help ===")
    print(await repl.process("/help", verbose=False))

    print(f"\n{'=' * 60}")
    print("\n=== /tools ===")
    print(await repl.process("/tools", verbose=False))

    print(f"\n{'=' * 60}")
    print("\n=== 일반 메시지 (mock) ===")
    result = await repl.process("안녕!", verbose=False)
    print(f"응답: {result}")

    print(f"\n{'=' * 60}")
    print("\n=== /cost ===")
    print(await repl.process("/cost", verbose=False))

    print(f"\n{'=' * 60}")
    print("\nAPI 키를 설정하면 실제 도구 호출을 포함한 대화를 테스트할 수 있습니다.")
    print("  export ANTHROPIC_API_KEY='sk-...'")
    print("  export OPENAI_API_KEY='sk-...'")

### 입력 분기 동작 확인

REPL의 핵심인 `parse_input()`의 동작을 직접 확인합니다.
모든 사용자 입력은 이 분기를 통과합니다.

In [ ]:
# --- 입력 파싱 테스트 ---
print("=== parse_input() 동작 확인 ===\n")

test_inputs = [
    "/help",                    # 커맨드 (정확한 이름)
    "/h",                       # 커맨드 (별칭)
    "/?",                       # 커맨드 (별칭)
    "/cost",                    # 커맨드
    "/clear",                   # 커맨드
    "/cl",                      # 커맨드 (별칭)
    "/unknown_cmd",             # 알 수 없는 커맨드
    "안녕하세요",               # 일반 메시지
    "파일 목록을 보여줘",       # 일반 메시지
    "",                         # 빈 입력
]

for text in test_inputs:
    cmd, args = repl.parse_input(text)
    if cmd is None:
        cmd_info = "-> QueryEngine (일반 메시지)"
    else:
        cmd_type = "Local" if isinstance(cmd, LocalCommand) else "Prompt"
        cmd_info = f"-> /{cmd.name} [{cmd_type}]"
    print(f"  {text!r:30s} {cmd_info}")

print()
print("핵심: '/'가 있으면 커맨드, 없으면 LLM으로 전달.")
print("이 단순한 분기가 Claude Code의 전체 입력 처리 흐름입니다.")

### 권한 시스템 통합 확인

MiniQueryEngine이 도구 실행 전에 PermissionSystem을 통해 권한을 체크하는 과정을 확인합니다.
이것은 Claude Code의 `src/services/tools/toolExecution.ts`에서 `checkPermissions()` → `tool.execute()` 흐름에 대응합니다.

In [ ]:
# --- 권한 체크가 엔진에서 어떻게 동작하는지 시뮬레이션 ---

print("=== 권한 시스템 통합 테스트 ===\n")

# 엔진의 권한 시스템이 다양한 도구 호출에 어떻게 반응하는지 확인
test_cases = [
    ("Read",  {"file_path": "/tmp/test.py"},    True,  "읽기 도구 -> 기본 허용"),
    ("Bash",  {"command": "git status"},         False, "안전한 git 명령 -> 규칙 허용"),
    ("Bash",  {"command": "rm -rf /important"},  False, "위험한 명령 -> 규칙 거부"),
    ("Bash",  {"command": "pip install flask"},  False, "일반 bash -> ASK (비대화형=거부)"),
]

for tool_name, tool_input, is_ro, description in test_cases:
    result = await engine.permissions.check_permission(
        tool_name, tool_input, is_read_only=is_ro
    )
    status = {
        "allowed": "[허용]",
        "denied":  "[거부]",
        "ask_user": "[확인필요]",
    }.get(result.result.value, result.result.value)

    print(f"  {status} {description}")
    print(f"         도구={tool_name}, 입력={tool_input}")
    print(f"         이유: {result.reason}")
    print()

print("핵심: 도구 실행 전에 반드시 권한 체크를 거칩니다.")
print("이 파이프라인 덕분에 에이전트가 위험한 작업을 자동으로 수행하지 못합니다.")

---

## 핵심 설계 원칙 정리

### 1. 입력 분기가 전부다
REPL의 핵심은 `/`로 시작하면 커맨드, 아니면 LLM. 이 단순한 분기가 전체 시스템의 진입점입니다. Claude Code의 `processInput()` (src/ink/components/App.tsx:309)도 본질적으로 이 패턴입니다. 복잡한 시스템의 입구는 놀랍도록 단순합니다.

### 2. 중간 통합은 조기 검증
13개 서브시스템을 다 만든 후 통합하면 디버깅이 어렵습니다. 핵심 부품(LLMClient, ToolRegistry, PermissionSystem)만으로 먼저 동작을 확인하고, 이후 Step에서 점진적으로 기능을 추가합니다. 이것은 소프트웨어 개발의 "Continuous Integration" 원칙과 같습니다.

### 3. Dependency Injection
MiniQueryEngine은 client, registry, permissions를 외부에서 주입받습니다. 이 덕분에:
- **Mock 클라이언트**로 API 없이 테스트할 수 있습니다
- **다른 권한 모드**로 쉽게 교체할 수 있습니다 (AUTO, ASK, DENY)
- **도구를 추가/제거**해도 엔진 코드를 수정할 필요가 없습니다

Claude Code의 `assembleToolPool()` (src/tools.ts:345)이 바로 이 Dependency Injection 패턴을 사용합니다.

### 4. REPL = 루프 안의 루프
외부 루프(REPL: 사용자 입력 반복)가 내부 루프(agent_loop: API 호출 반복)를 감싸는 구조입니다:

```
while (사용자 입력을 받는다) {          ← 외부 루프 (MiniREPL.run)
    while (도구 호출이 필요하다) {       ← 내부 루프 (MiniQueryEngine.submit)
        API 호출 → 도구 실행 → 결과 추가
    }
    최종 응답 표시
}
```

Claude Code의 전체 아키텍처가 이 이중 루프입니다. 모든 복잡한 기능(컨텍스트 압축, 메모리, 스킬 등)은 내부 루프의 각 단계에 플러그인으로 끼워 넣어집니다.

---

## 다음 Step 미리보기

핵심 부품으로 동작하는 미니 에이전트를 만들었습니다. 하지만 대화가 길어지면 컨텍스트 윈도우를 초과하고, 세션이 끝나면 모든 맥락이 사라집니다.

다음 Step에서는:
- **Step 9: 메모리** -- 세션 간 지식 유지, CLAUDE.md에서 프로젝트 맥락 로드, 세션 저장/복구
- **Step 10: 스킬 시스템** -- 재사용 가능한 워크플로우 정의와 SkillTool을 통한 LLM 호출

이 미니 REPL이 향후 Step에서 기능이 추가될 때의 **기준선(baseline)**이 됩니다. 새 기능이 추가되어도 기본 흐름(입력 분기 -> 에이전트 루프 -> 도구 실행 -> 응답)은 변하지 않습니다. 변하는 것은 루프 안에 끼워 넣는 플러그인뿐입니다.